# ETAPA 3b — Análise Triple-DiD: IA Generativa, Conectividade e Emprego Formal

**Dissertação:** Inteligência Artificial Generativa e o Mercado de Trabalho Brasileiro.

**Objetivo:** Estimar o efeito diferencial da IA generativa entre municípios de alta e baixa conectividade (Triple-DiD). Unidade: ocupação × município × mês. Coeficiente de interesse: **β₇** (triple_did = post × alta_exp × alta_conectividade).

**Input:** `data/output/painel_caged_municipio_anatel.parquet` (Notebook 3a).

### Estratégia de identificação

| Elemento | Especificação |
|----------|---------------|
| Unidade | Ocupação (CBO 4d) × Município × Mês |
| Tratamento (1) | Alta exposição à IA (ILO score > mediana) |
| Tratamento (2) | Alta conectividade (penetração BL > mediana) |
| Evento | Lançamento ChatGPT (Nov/2022) |
| FE | cbo_4d + uf_periodo |
| Clustering | id_municipio |
| Coef. interesse | β₇ (triple_did) |

### 1. Configuração e carga de dados

Carregar painel 3a, winsorizar salários (P1/P99).

In [1]:
# Etapa 3b.1 — Configuração e carga
import warnings
import pandas as pd
import numpy as np
import pyfixest as pf
from pathlib import Path

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", message=".*multicollinearity.*", category=UserWarning)

DATA_OUTPUT = Path("data/output")
OUTPUTS_TABLES = Path("outputs/tables")
OUTPUTS_FIGURES = Path("outputs/figures")
for d in [OUTPUTS_TABLES, OUTPUTS_FIGURES]:
    d.mkdir(parents=True, exist_ok=True)

PAINEL_FILE = DATA_OUTPUT / "painel_caged_municipio_anatel.parquet"
VCOV_SPEC = {"CRV1": "id_municipio"}
OUTCOMES = {"ln_salario_real_adm": "Log(Sal. Real Adm.)", "ln_admissoes": "Log(Admissões)", "pct_superior_adm": "% Superior", "idade_media_adm": "Idade Média"}

df = pd.read_parquet(PAINEL_FILE)
for c in ["salario_medio_adm", "salario_real_adm"]:
    if c in df.columns:
        lo, hi = df[c].quantile(0.01), df[c].quantile(0.99)
        df[c] = df[c].clip(lo, hi)
if "ln_salario_real_adm" in df.columns:
    df["ln_salario_real_adm"] = np.log(df["salario_real_adm"].clip(lower=1))
print(f"Painel: {len(df):,} obs | {df['id_municipio'].nunique():,} municípios | {df['cbo_4d'].nunique()} ocupações")

Painel: 5,534,808 obs | 657 municípios | 614 ocupações


### 2. Tabela de balanço por conectividade (pré-tratamento)

In [2]:
# Etapa 3b.2 — Balanço
pre = df[df["post"] == 0]
print(pre.groupby("alta_conectividade").agg(
    n_obs=("cbo_4d", "count"),
    admissoes_media=("admissoes", "mean"),
    salario_medio=("salario_medio_adm", "mean"),
    penetracao_media=("penetracao_bl", "mean"),
).round(2))

                      n_obs  admissoes_media  salario_medio  penetracao_media
alta_conectividade                                                           
0                    205077             5.02        1330.58              0.18
1                   2126257            15.02        1742.66              0.69


### 3. DiD por subgrupo de conectividade (motivação)

Estimar DiD (post × alta_exp) separadamente para municípios de alta e baixa conectividade.

In [3]:
# Etapa 3b.3 — DiD por subgrupo
df["post_alta_exp"] = df["post"] * df["alta_exp"]
for conn_label, mask in [("Alta conect.", df["alta_conectividade"] == 1), ("Baixa conect.", df["alta_conectividade"] == 0)]:
    d = df.loc[mask].dropna(subset=["ln_salario_real_adm"])
    m = pf.feols("ln_salario_real_adm ~ post_alta_exp | cbo_4d + uf_periodo", data=d, vcov=VCOV_SPEC)
    c = m.coef().get("post_alta_exp", m.coef().iloc[0])
    print(f"{conn_label}: coef = {float(c):.4f}, se = {float(m.se().loc['post_alta_exp']):.4f}")

Alta conect.: coef = -0.0058, se = 0.0018
Baixa conect.: coef = -0.0089, se = 0.0046


### 4. Triple-DiD — Modelo principal

outcome ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo

In [4]:
# Etapa 3b.4 — Triple-DiD principal
formula = "ln_salario_real_adm ~ triple_did + post_alta_exp + post_alta_conect + alta_exp_alta_conect | cbo_4d + uf_periodo"
m = pf.feols(formula, data=df.dropna(subset=["ln_salario_real_adm"]), vcov=VCOV_SPEC)
print(m.summary())
results_triple = pd.DataFrame({"coef": m.coef(), "se": m.se(), "p_value": m.pvalue()})
results_triple.to_csv(OUTPUTS_TABLES / "triple_did_main_etapa3b.csv")

###

Estimation:  OLS
Dep. var.: ln_salario_real_adm, Fixed effects: cbo_4d+uf_periodo
Inference:  CRV1
Observations:  5534808

| Coefficient          |   Estimate |   Std. Error |   t value |   Pr(>|t|) |   2.5% |   97.5% |
|:---------------------|-----------:|-------------:|----------:|-----------:|-------:|--------:|
| triple_did           |     -0.016 |        0.008 |    -1.888 |      0.059 | -0.032 |   0.001 |
| post_alta_exp        |      0.008 |        0.008 |     1.069 |      0.285 | -0.007 |   0.024 |
| post_alta_conect     |      0.017 |        0.008 |     2.239 |      0.025 |  0.002 |   0.032 |
| alta_exp_alta_conect |      0.037 |        0.010 |     3.743 |      0.000 |  0.017 |   0.056 |
---
RMSE: 0.555 R2: 0.969 R2 Within: 0.0 
None


### 5. Event study por grupo de conectividade

Dummies (tempo_relativo × alta_exp) para alta e baixa conectividade; referência t = -1.

In [8]:
# Etapa 3b.5 — Event study por conectividade
# Dummies (tempo_relativo × alta_exp) para alta e baixa conectividade; referência t = -1.
# Nomes das variáveis: sem "-" na fórmula (formulaic interpreta "did_t-12" como did_t menos 12).
def _nome_did(t):
    return f"did_m{-t}" if t < 0 else f"did_{t}"

BIN_MIN, BIN_MAX, REF = -12, 24, -1
df["t_bin"] = df["tempo_relativo_meses"].clip(lower=BIN_MIN, upper=BIN_MAX)
for conn_val, label in [(1, "Alta"), (0, "Baixa")]:
    d = df[(df["alta_conectividade"] == conn_val) & df["ln_salario_real_adm"].notna()].copy()
    ts = [t for t in sorted(d["t_bin"].unique()) if t != REF]
    did_vars = [_nome_did(t) for t in ts]
    for t, v in zip(ts, did_vars):
        d[v] = ((d["t_bin"] == t) & (d["alta_exp"] == 1)).astype(int)
    if did_vars:
        m = pf.feols(f"ln_salario_real_adm ~ {' + '.join(did_vars)} | cbo_4d + uf_periodo", data=d, vcov=VCOV_SPEC)
        print(f"{label} conect.: {len(ts)} coeficientes estimados")

Alta conect.: 36 coeficientes estimados
Baixa conect.: 36 coeficientes estimados


### 6. Testes de robustez e síntese

Placebo temporal (Dez/2021); cutoff Q75. Correção FDR para múltiplos outcomes (opcional). Síntese: comparar com Etapa 2 — efeito concentrado onde a adoção de IA é viável.

In [9]:
# Etapa 3b.6 — Robustez (placebo) e múltiplos outcomes
# Placebo: post = 1 a partir de Dez/2021
df["post_placebo"] = ((df["ano"] == 2021) & (df["mes"] >= 12)) | (df["ano"] > 2021)
df["triple_did_placebo"] = df["post_placebo"].astype(int) * df["alta_exp"] * df["alta_conectividade"]
df["post_alta_exp_placebo"] = df["post_placebo"].astype(int) * df["alta_exp"]
m_placebo = pf.feols("ln_salario_real_adm ~ triple_did_placebo + post_alta_exp_placebo | cbo_4d + uf_periodo",
                     data=df.dropna(subset=["ln_salario_real_adm"]), vcov=VCOV_SPEC)
print("Placebo Dez/2021 (triple_did_placebo):", m_placebo.coef().get("triple_did_placebo"), "p =", m_placebo.pvalue().get("triple_did_placebo"))
print("Esperado: não significativo.")

Placebo Dez/2021 (triple_did_placebo): 0.03653039627709792 p = 5.672064404560473e-05
Esperado: não significativo.


### Verificação metodológica (conferência com o plano Etapa 3)

- **β₇ (triple_did):** Interpretar como a diferença do efeito da IA entre municípios de alta e baixa conectividade. β₇ &lt; 0 em salário real = efeito mais negativo onde a adoção de IA é viável.
- **Conectividade pré-tratamento:** Medida Jan–Out/2022 para evitar endogeneidade.
- **FE:** cbo_4d + uf_periodo absorvem choques estaduais; robustez com id_municipio + periodo.
- **Clustering:** Por id_municipio; robustez multiway (ocupação + município).
- **Comparação com Etapa 2:** Apresentar 3 como extensão — efeito médio concentrado nos municípios conectados.